# Single-Run Summary

Workflow: edit the function → run the show cell to preview → run the save cell.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def find_project_root():
    path = Path.cwd()
    for _ in range(5):
        if (path / "code").is_dir() and (path / "results").is_dir():
            return path
        path = path.parent
    return Path.cwd()


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Output  : {RESULTS_DIR}")

In [ ]:
# -- load data -------------------------------------------------------------

csv_path = RESULTS_DIR / "housing_abm_results.csv"
if not csv_path.exists():
    raise FileNotFoundError(f"{csv_path} not found. Run code/run.py first.")

df = pd.read_csv(csv_path, index_col=0)
print(f"Loaded {len(df)} rows x {len(df.columns)} cols")
print(f"Columns: {list(df.columns)}")

In [ ]:
# -- helper ----------------------------------------------------------------


def _stacked_share_plot(ax, data, columns, title, colors, window=12):
    smoothed = data[list(columns)].rolling(window, min_periods=1, center=True).mean()
    smoothed.plot.area(ax=ax, stacked=True, color=colors, alpha=0.8)
    ax.set_title(title)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Share")
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1], loc="upper left", fontsize=8)

---
## Figure 1: Market Prices


In [ ]:
# 1a. Function


def plot_market_prices(df, figsize=(12, 4)):
    """1x2 grid: sale price and rent."""
    df_filled = df.copy()
    for col in ("avg_sale_price", "avg_rent"):
        if col in df_filled.columns:
            med = float(df_filled[col].median(skipna=True)) if df_filled[col].notna().any() else 0.0
            df_filled[col + "_filled"] = df_filled[col].ffill().fillna(med)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    if "avg_sale_price_filled" in df_filled.columns:
        df_filled["avg_sale_price_filled"].plot(ax=ax1, title="Average Sale Price (filled)")
    elif "avg_sale_price" in df.columns:
        df["avg_sale_price"].ffill().plot(ax=ax1, title="Average Sale Price")
    else:
        ax1.text(0.5, 0.5, "avg_sale_price not collected", ha="center", transform=ax1.transAxes)

    if "avg_rent_filled" in df_filled.columns:
        df_filled["avg_rent_filled"].plot(ax=ax2, title="Average Rent (filled)")
    elif "avg_rent" in df.columns:
        df["avg_rent"].ffill().plot(ax=ax2, title="Average Rent")
    else:
        ax2.text(0.5, 0.5, "avg_rent not collected", ha="center", transform=ax2.transAxes)

    fig.tight_layout()
    return fig

In [ ]:
# 1b. Show
fig1 = plot_market_prices(df)
plt.show()

In [ ]:
# 1c. Save
fig1.savefig(RESULTS_DIR / "figs" / "market_prices.png", dpi=300, bbox_inches="tight")
plt.close(fig1)
print("Saved: market_prices.png")

---
## Figure 2: Housing Stock Ownership


In [ ]:
# 2a. Function


def plot_stock_ownership(df, figsize=(7, 4)):
    """Stacked area: owner-occupier, landlord, institutional shares of housing stock."""
    stock_cols = [
        c
        for c in (
            "owner_occupier_ownership_share",
            "landlord_ownership_share",
            "institutional_ownership_share",
        )
        if c in df.columns
    ]

    fig, ax = plt.subplots(figsize=figsize)

    if len(stock_cols) == 3:
        _stacked_share_plot(
            ax,
            df,
            stock_cols,
            "Housing Stock Ownership by Kind",
            colors=["#4c72b0", "#dd8452", "#55a868"],
        )
    else:
        ax.text(0.5, 0.5, "Stock share data not available", ha="center", transform=ax.transAxes)

    fig.tight_layout()
    return fig

In [ ]:
# 2b. Show
fig2 = plot_stock_ownership(df)
plt.show()

In [ ]:
# 2c. Save
fig2.savefig(RESULTS_DIR / "figs" / "stock_ownership.png", dpi=300, bbox_inches="tight")
plt.close(fig2)
print("Saved: stock_ownership.png")

---
## Figure 3: Marginal Pricer Dynamics


In [ ]:
# 3a. Function


def plot_marginal_pricer(df, figsize=(12, 4)):
    """1x2 grid: marginal pricer by count and by value."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)

    mp_count = ["owner_occupier_share", "landlord_share", "institution_share"]
    if all(c in df.columns for c in mp_count):
        _stacked_share_plot(
            ax1,
            df,
            mp_count,
            "Marginal Pricer Share (count)",
            colors=["#4c72b0", "#dd8452", "#55a868"],
        )
    else:
        ax1.text(
            0.5, 0.5, "Marginal pricer (count) not available", ha="center", transform=ax1.transAxes
        )

    mp_value = ["owner_occupier_value_share", "landlord_value_share", "institution_value_share"]
    if all(c in df.columns for c in mp_value):
        _stacked_share_plot(
            ax2,
            df,
            mp_value,
            "Marginal Pricer Share (value)",
            colors=["#4c72b0", "#dd8452", "#55a868"],
        )
    else:
        ax2.text(
            0.5, 0.5, "Marginal pricer (value) not available", ha="center", transform=ax2.transAxes
        )

    fig.tight_layout()
    return fig

In [ ]:
# 3b. Show
fig3 = plot_marginal_pricer(df)
plt.show()

In [ ]:
# 3c. Save
fig3.savefig(RESULTS_DIR / "figs" / "marginal_pricer.png", dpi=300, bbox_inches="tight")
plt.close(fig3)
print("Saved: marginal_pricer.png")